# Extract Metadata Demo

This notebook demonstrates how to use the Curator workflows client library to extract metadata from documents.

## Setup and Configuration

In [ ]:
import sys
import os
import glob

from dotenv import load_dotenv

# Import the client library
sys.path.append('../../src')
from extract_client import ExtractMetadataClient
from utils import (
    detailed_status_callback, 
    batch_status_callback,
    extract_entity_filenames_from_csv,
    load_entity_filenames_from_directory,
    map_files_to_entities,
    display_extraction_results
)

# Load environment variables
load_dotenv()

# Create client
client = ExtractMetadataClient()

print(f"API URL: {client.api_url}")
print(f"API Key configured: {'Yes' if client.api_key else 'No'}")

## 1. Basic Extraction from a Local File

### 1.1 Define File Paths and Load Entity Filenames

In [ ]:
# Define the file path and ground truth path
file_path = "./data/nf_8.pdf"
ground_truth_path = "./data/nf_8.csv"

# Check if file exists
if not os.path.exists(file_path):
    print(f"❌ File not found at: {file_path}")
else:
    print(f"📄 Found file: {file_path}")
    print(f"File size: {os.path.getsize(file_path)} bytes")

# Extract entity filenames from the ground truth file
entity_filenames = extract_entity_filenames_from_csv(ground_truth_path)

### 1.2 Extract Metadata from a Local File

In [ ]:
print("🔍 Extracting metadata from document with entity filenames...")

# Define the schema key for the schema file in S3
schema_key = "extract_metadata/nf/schema.json"

# Using the extract_metadata function with source='local'
result = client.extract_metadata(
    source='local',
    file_path=file_path,
    schema_name="nf",
    schema_key=schema_key,  # Explicitly provide the schema key
    filenames=entity_filenames,
    max_attempts=20,
    interval=20,
    output_path="extracted_metadata.csv",
    callback=detailed_status_callback
)

### 1.3 Display the Results

In [ ]:
# Display the status and detailed extraction results
display_extraction_results(result, entity_filenames)